In [1]:
import pandas as pd
import numpy as np

results = pd.read_parquet(
    "../../data/xfp_test_predictions_2025.parquet"
)

results.shape

(1122, 16)

In [2]:
results.columns.tolist()

['game_id',
 'posteam',
 'player_id',
 'player_name',
 'target_share',
 'red_zone_share',
 'goal_line_share',
 'carries',
 'targets',
 'opportunities',
 'total_air_yards',
 'avg_air_yards',
 'avg_yardline_100',
 'actual_fp_game',
 'xFP',
 'regression_delta']

In [3]:
player_metrics = (
    results
    .groupby(["player_id", "player_name", "posteam"])
    .agg(
        actual_fp=("actual_fp_game", "sum"),
        xFP=("xFP", "sum"),
        games=("game_id", "nunique"),
        opportunities=("opportunities", "sum"),
        targets=("targets", "sum"),
        carries=("carries", "sum")
    )
    .reset_index()
)

player_metrics.head()

,player_id,player_name,posteam,actual_fp,xFP,games,opportunities,targets,carries
0,00-0022942,P.Rivers,IND,0.0,0.280942,1,1.0,0,1.0
1,00-0023459,A.Rodgers,PIT,3.2,1.251397,3,4.0,0,4.0
2,00-0026300,J.Johnson,WAS,0.0,0.318035,1,1.0,0,1.0
3,00-0026498,M.Stafford,LA,1.8,2.124500,2,3.0,0,3.0
4,00-0028118,T.Taylor,NYJ,4.9,3.096000,1,7.0,0,7.0


In [4]:
player_metrics["regression_delta"] = (
    player_metrics["actual_fp"] - player_metrics["xFP"]
)

player_metrics[
    [
        "player_name",
        "posteam",
        "actual_fp",
        "xFP",
        "regression_delta"
    ]
].head(10)

,player_name,posteam,actual_fp,xFP,regression_delta
0,P.Rivers,IND,0.0,0.280942,-0.280942
1,A.Rodgers,PIT,3.2,1.251397,1.948603
2,J.Johnson,WAS,0.0,0.318035,-0.318035
3,M.Stafford,LA,1.8,2.124500,-0.324500
4,T.Taylor,NYJ,4.9,3.096000,1.804000
5,R.Wilson,NYG,1.2,4.050000,-2.850000
6,K.Juszczyk,SF,1.8,1.593000,0.207000
7,A.Thielen,MIN,0.0,7.530500,-7.530500
8,Z.Ertz,WAS,32.1,24.522233,7.577767
9,K.Allen,LAC,28.1,30.977995,-2.877995


In [5]:
def classify_signal(delta):
    if delta <= -10:
        return "STRONG BUY"
    elif delta <= -5:
        return "BUY"
    elif delta >= 10:
        return "STRONG SELL"
    elif delta >= 5:
        return "SELL"
    else:
        return "HOLD"

player_metrics["signal"] = player_metrics["regression_delta"].apply(classify_signal)

player_metrics[
    ["player_name", "actual_fp", "xFP", "regression_delta", "signal"]
].head(15)

,player_name,actual_fp,xFP,regression_delta,signal
0,P.Rivers,0.0,0.280942,-0.280942,HOLD
1,A.Rodgers,3.2,1.251397,1.948603,HOLD
2,J.Johnson,0.0,0.318035,-0.318035,HOLD
3,M.Stafford,1.8,2.124500,-0.324500,HOLD
4,T.Taylor,4.9,3.096000,1.804000,HOLD
5,R.Wilson,1.2,4.050000,-2.850000,HOLD
6,K.Juszczyk,1.8,1.593000,0.207000,HOLD
7,A.Thielen,0.0,7.530500,-7.530500,BUY
8,Z.Ertz,32.1,24.522233,7.577767,SELL
9,K.Allen,28.1,30.977995,-2.877995,HOLD


In [6]:
training_data = pd.read_parquet(
    "../../data/training_2025_player_games.parquet"
)

training_data.shape

(5610, 14)

In [7]:
import joblib

model = joblib.load(
    "../../ml/models/xfp_random_forest_v1.joblib"
)

features = [
    "target_share",
    "red_zone_share",
    "goal_line_share",
    "carries",
    "targets",
    "opportunities",
    "total_air_yards",
    "avg_air_yards",
    "avg_yardline_100"
]

training_data["xFP"] = model.predict(
    training_data[features]
)

training_data["regression_delta"] = (
    training_data["actual_fp_game"] - training_data["xFP"]
)

training_data[
    ["player_name", "actual_fp_game", "xFP", "regression_delta"]
].head()

,player_name,actual_fp_game,xFP,regression_delta
0,J.Conner,14.4,14.2475,0.1525
1,Z.Jones,1.4,0.8775,0.5225
2,K.Murray,3.8,4.5390,-0.7390
3,G.Dortch,0.8,1.0205,-0.2205
4,T.McBride,12.1,12.7780,-0.6780


In [8]:
season_metrics = (
    training_data
    .groupby(["player_id", "player_name", "posteam"])
    .agg(
        actual_fp=("actual_fp_game", "sum"),
        xFP=("xFP", "sum"),
        games=("game_id", "nunique"),
        opportunities=("opportunities", "sum"),
        targets=("targets", "sum"),
        carries=("carries", "sum")
    )
    .reset_index()
)

season_metrics["regression_delta"] = (
    season_metrics["actual_fp"] - season_metrics["xFP"]
)

season_metrics.head()

,player_id,player_name,posteam,actual_fp,xFP,games,opportunities,targets,carries,regression_delta
0,00-0022942,P.Rivers,IND,0.0,0.280942,1,1.0,0,1.0,-0.280942
1,00-0023459,A.Rodgers,PIT,12.9,8.870413,12,14.0,1,13.0,4.029587
2,00-0026158,J.Flacco,CIN,9.8,9.219912,5,7.0,0,7.0,0.580088
3,00-0026158,J.Flacco,CLE,1.4,1.835957,3,5.0,0,5.0,-0.435957
4,00-0026300,J.Johnson,WAS,11.6,10.807285,3,11.0,0,11.0,0.792715


In [10]:
def classify_signal(delta):
    if delta <= -20:
        return "STRONG BUY"
    elif delta <= -10:
        return "BUY"
    elif delta >= 20:
        return "STRONG SELL"
    elif delta >= 10:
        return "SELL"
    else:
        return "HOLD"

season_metrics["signal"] = (
    season_metrics["regression_delta"]
    .apply(classify_signal)
)

season_metrics[
    ["player_name", "actual_fp", "xFP", "regression_delta", "signal"]
].head(15)

,player_name,actual_fp,xFP,regression_delta,signal
0,P.Rivers,0.0,0.280942,-0.280942,HOLD
1,A.Rodgers,12.9,8.870413,4.029587,HOLD
2,J.Flacco,9.8,9.219912,0.580088,HOLD
3,J.Flacco,1.4,1.835957,-0.435957,HOLD
4,J.Johnson,11.6,10.807285,0.792715,HOLD
5,M.Stafford,4.0,6.552639,-2.552639,HOLD
6,A.Dalton,0.1,0.518052,-0.418052,HOLD
7,T.Taylor,20.5,17.330500,3.169500,HOLD
8,R.Wilson,10.6,14.295500,-3.695500,HOLD
9,K.Cousins,7.7,7.505767,0.194233,HOLD


In [11]:
season_metrics["actual_fp_per_game"] = (
    season_metrics["actual_fp"] / season_metrics["games"]
)

season_metrics["xFP_per_game"] = (
    season_metrics["xFP"] / season_metrics["games"]
)

season_metrics[
    [
        "player_name",
        "games",
        "actual_fp_per_game",
        "xFP_per_game",
        "regression_delta",
        "signal"
    ]
].head(15)

,player_name,games,actual_fp_per_game,xFP_per_game,regression_delta,signal
0,P.Rivers,1,0.000000,0.280942,-0.280942,HOLD
1,A.Rodgers,12,1.075000,0.739201,4.029587,HOLD
2,J.Flacco,5,1.960000,1.843982,0.580088,HOLD
3,J.Flacco,3,0.466667,0.611986,-0.435957,HOLD
4,J.Johnson,3,3.866667,3.602428,0.792715,HOLD
5,M.Stafford,10,0.400000,0.655264,-2.552639,HOLD
6,A.Dalton,1,0.100000,0.518052,-0.418052,HOLD
7,T.Taylor,5,4.100000,3.466100,3.169500,HOLD
8,R.Wilson,4,2.650000,3.573875,-3.695500,HOLD
9,K.Cousins,6,1.283333,1.250961,0.194233,HOLD


In [12]:
buy_low_season = season_metrics.sort_values(
    "regression_delta",
    ascending=True
)[
    [
        "player_name",
        "posteam",
        "actual_fp",
        "xFP",
        "actual_fp_per_game",
        "xFP_per_game",
        "regression_delta",
        "signal"
    ]
].head(15)

buy_low_season

,player_name,posteam,actual_fp,xFP,actual_fp_per_game,xFP_per_game,regression_delta,signal
207,J.Jeudy,CLE,122.7,154.931067,7.217647,9.113592,-32.231067,STRONG BUY
508,A.Mitchell,NYJ,65.7,94.443000,8.212500,11.805375,-28.743000,STRONG BUY
543,E.Ayomanor,TEN,122.5,149.680000,7.656250,9.355000,-27.180000,STRONG BUY
618,I.Bond,CLE,52.6,77.315756,3.506667,5.154384,-24.715756,STRONG BUY
196,D.Mooney,ATL,82.3,106.240700,5.486667,7.082713,-23.940700,STRONG BUY
459,K.Vidal,LAC,125.0,148.179900,9.615385,11.398454,-23.179900,STRONG BUY
294,J.Ford,CLE,43.6,65.985685,3.353846,5.075822,-22.385685,STRONG BUY
114,S.Barkley,PHI,248.4,268.679933,14.611765,15.804702,-20.279933,STRONG BUY
533,E.Egbuka,TB,193.7,213.127029,11.394118,12.536884,-19.427029,BUY
509,B.Thomas,JAX,148.9,166.661467,9.926667,11.110764,-17.761467,BUY


In [13]:
sell_high_season = season_metrics.sort_values(
    "regression_delta",
    ascending=False
)[
    [
        "player_name",
        "posteam",
        "actual_fp",
        "xFP",
        "actual_fp_per_game",
        "xFP_per_game",
        "regression_delta",
        "signal"
    ]
].head(15)

sell_high_season

,player_name,posteam,actual_fp,xFP,actual_fp_per_game,xFP_per_game,regression_delta,signal
187,J.Taylor,IND,362.3,316.682000,21.311765,18.628353,45.618000,STRONG SELL
369,J.Smith-Njigba,SEA,410.8,365.547024,20.540000,18.277351,45.252976,STRONG SELL
436,P.Nacua,LA,460.6,417.088500,24.242105,21.952026,43.511500,STRONG SELL
117,J.Allen,BUF,166.0,137.253742,9.222222,7.625208,28.746258,STRONG SELL
437,J.Gibbs,DET,368.6,340.873500,21.682353,20.051382,27.726500,STRONG SELL
91,D.Goedert,PHI,204.5,180.346400,12.781250,11.271650,24.153600,STRONG SELL
38,D.Henry,BAL,285.5,261.790500,16.794118,15.399441,23.709500,STRONG SELL
612,T.Henderson,NE,224.1,201.241287,10.671429,9.582918,22.858713,STRONG SELL
423,T.Kraft,GB,117.2,95.555400,14.650000,11.944425,21.644600,STRONG SELL
415,D.Kincaid,BUF,158.2,139.268500,11.300000,9.947750,18.931500,SELL


In [14]:
season_metrics["regression_delta_per_game"] = (
    season_metrics["regression_delta"] / season_metrics["games"]
)

def classify_signal_pg(delta):
    if delta <= -2.0:
        return "STRONG BUY"
    elif delta <= -1.0:
        return "BUY"
    elif delta >= 2.0:
        return "STRONG SELL"
    elif delta >= 1.0:
        return "SELL"
    else:
        return "HOLD"

season_metrics["signal"] = (
    season_metrics["regression_delta_per_game"]
    .apply(classify_signal_pg)
)

season_metrics[
    [
        "player_name",
        "actual_fp_per_game",
        "xFP_per_game",
        "regression_delta_per_game",
        "signal"
    ]
].head(15)

,player_name,actual_fp_per_game,xFP_per_game,regression_delta_per_game,signal
0,P.Rivers,0.000000,0.280942,-0.280942,HOLD
1,A.Rodgers,1.075000,0.739201,0.335799,HOLD
2,J.Flacco,1.960000,1.843982,0.116018,HOLD
3,J.Flacco,0.466667,0.611986,-0.145319,HOLD
4,J.Johnson,3.866667,3.602428,0.264238,HOLD
5,M.Stafford,0.400000,0.655264,-0.255264,HOLD
6,A.Dalton,0.100000,0.518052,-0.418052,HOLD
7,T.Taylor,4.100000,3.466100,0.633900,HOLD
8,R.Wilson,2.650000,3.573875,-0.923875,HOLD
9,K.Cousins,1.283333,1.250961,0.032372,HOLD


In [6]:
season_metrics = (
    training_data
    .groupby(["player_id", "player_name", "posteam"])
    .agg(
        actual_fp=("actual_fp_game", "sum"),
        xFP=("xFP", "sum"),
        games=("game_id", "nunique"),
        opportunities=("opportunities", "sum"),
        targets=("targets", "sum"),
        carries=("carries", "sum")
    )
    .reset_index()
)

season_metrics["regression_delta"] = (
    season_metrics["actual_fp"] - season_metrics["xFP"]
)

NameError: name 'training_data' is not defined

In [7]:
import pandas as pd
import joblib

In [8]:
training_data = pd.read_parquet(
    "../../data/training_2025_player_games.parquet"
)

In [9]:
model = joblib.load(
    "../../ml/models/xfp_random_forest_v1.joblib"
)

features = [
    "target_share",
    "red_zone_share",
    "goal_line_share",
    "carries",
    "targets",
    "opportunities",
    "total_air_yards",
    "avg_air_yards",
    "avg_yardline_100"
]

training_data["xFP"] = model.predict(training_data[features])

training_data["regression_delta"] = (
    training_data["actual_fp_game"] - training_data["xFP"]
)

In [10]:
import pandas as pd
import joblib

training_data = pd.read_parquet(
    "../../data/training_2025_player_games.parquet"
)

model = joblib.load(
    "../../ml/models/xfp_random_forest_v1.joblib"
)

features = [
    "target_share",
    "red_zone_share",
    "goal_line_share",
    "carries",
    "targets",
    "opportunities",
    "total_air_yards",
    "avg_air_yards",
    "avg_yardline_100"
]

training_data["xFP"] = model.predict(training_data[features])

training_data["regression_delta"] = (
    training_data["actual_fp_game"] - training_data["xFP"]
)

training_data.head()

,game_id,posteam,player_id,player_name,target_share,red_zone_share,goal_line_share,carries,targets,opportunities,total_air_yards,avg_air_yards,avg_yardline_100,actual_fp_game,xFP,regression_delta
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,0.137931,0.333333,0.666667,12.0,4,16.0,-15.0,-3.750000,54.187500,14.4,14.2475,0.1525
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,0.034483,0.000000,0.000000,0.0,1,1.0,2.0,2.000000,37.000000,1.4,0.8775,0.5225
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,0.000000,0.111111,0.000000,7.0,0,7.0,0.0,0.000000,46.428571,3.8,4.5390,-0.7390
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,0.034483,0.000000,0.000000,0.0,1,1.0,-2.0,-2.000000,30.000000,0.8,1.0205,-0.2205
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,0.310345,0.111111,0.000000,0.0,9,9.0,35.0,3.888889,55.444444,12.1,12.7780,-0.6780


In [11]:
season_metrics = (
    training_data
    .groupby(["player_id", "player_name", "posteam"])
    .agg(
        actual_fp=("actual_fp_game", "sum"),
        xFP=("xFP", "sum"),
        games=("game_id", "nunique"),
        opportunities=("opportunities", "sum"),
        targets=("targets", "sum"),
        carries=("carries", "sum")
    )
    .reset_index()
)

season_metrics["regression_delta"] = (
    season_metrics["actual_fp"] - season_metrics["xFP"]
)

season_metrics["actual_fp_per_game"] = (
    season_metrics["actual_fp"] / season_metrics["games"]
)

season_metrics["xFP_per_game"] = (
    season_metrics["xFP"] / season_metrics["games"]
)

season_metrics["regression_delta_per_game"] = (
    season_metrics["regression_delta"] / season_metrics["games"]
)

season_metrics.head()

,player_id,player_name,posteam,actual_fp,xFP,games,opportunities,targets,carries,regression_delta,actual_fp_per_game,xFP_per_game,regression_delta_per_game
0,00-0022942,P.Rivers,IND,0.0,0.280942,1,1.0,0,1.0,-0.280942,0.000000,0.280942,-0.280942
1,00-0023459,A.Rodgers,PIT,12.9,8.870413,12,14.0,1,13.0,4.029587,1.075000,0.739201,0.335799
2,00-0026158,J.Flacco,CIN,9.8,9.219912,5,7.0,0,7.0,0.580088,1.960000,1.843982,0.116018
3,00-0026158,J.Flacco,CLE,1.4,1.835957,3,5.0,0,5.0,-0.435957,0.466667,0.611986,-0.145319
4,00-0026300,J.Johnson,WAS,11.6,10.807285,3,11.0,0,11.0,0.792715,3.866667,3.602428,0.264238


In [12]:
def classify_signal_pg(delta):
    if delta <= -2.0:
        return "STRONG BUY"
    elif delta <= -1.0:
        return "BUY"
    elif delta >= 2.0:
        return "STRONG SELL"
    elif delta >= 1.0:
        return "SELL"
    else:
        return "HOLD"

season_metrics["signal"] = (
    season_metrics["regression_delta_per_game"]
    .apply(classify_signal_pg)
)

season_metrics.to_parquet(
    "../../data/player_metrics_2025.parquet",
    index=False
)

In [13]:
type(season_metrics)

pandas.DataFrame